# arXiv Category Pulse

This notebook counts real arXiv submissions by category and decomposes the resulting monthly count series.

Use it for articles such as:

- Which AI fields are actually accelerating?
- Is `cs.CL` still rising after the LLM boom?
- Are biology and finance categories showing durable AI-adjacent growth?

Data source: arXiv API. No synthetic fallback.


In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Prefer the checkout when this notebook is run inside the repository.
repo_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / "src" / "detime").exists():
        repo_root = candidate
        break
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from examples.hot_trends.data import (
    HotTrendDataError,
    append_real_snapshot,
    build_arxiv_monthly_counts,
    fetch_coingecko_market_chart,
    fetch_defillama_stablecoin_chains,
    fetch_github_repo_metadata,
    fetch_github_stargazers,
    fetch_huggingface_models,
    fetch_wikipedia_pageviews,
    source_audit_table,
)
from examples.hot_trends.decomposition import (
    component_summary,
    decompose_table,
    editorial_priority,
    residual_event_table,
)
from examples.hot_trends.scoring import article_language_guardrails

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

CACHE_DIR = repo_root / "examples" / "hot_trends" / "cache"
OUTPUT_DIR = repo_root / "examples" / "hot_trends" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(repo_root)}")
    return path


## 1. Define the category watchlist


In [ ]:
queries = {
    "cs.AI": "cat:cs.AI",
    "cs.LG": "cat:cs.LG",
    "cs.CL": "cat:cs.CL",
    "cs.CV": "cat:cs.CV",
    "stat.ML": "cat:stat.ML",
    "q-bio.QM": "cat:q-bio.QM",
    "q-fin.ST": "cat:q-fin.ST",
    "econ.EM": "cat:econ.EM",
}
watchlist = pd.DataFrame([{"series": k, "query": v} for k, v in queries.items()])
watchlist


## 2. Fetch real monthly counts

The default window is intentionally modest so the notebook can run without hammering the API. Increase the window for a full article.


In [ ]:
START_MONTH = "2025-01-01"
END_MONTH = "2026-05-01"
SLEEP_SECONDS = 3.0

cache_path = CACHE_DIR / f"arxiv_category_counts_{START_MONTH}_{END_MONTH}.csv"
if cache_path.exists():
    counts = pd.read_csv(cache_path, parse_dates=["month"])
else:
    counts = build_arxiv_monthly_counts(queries, start_month=START_MONTH, end_month=END_MONTH, sleep_seconds=SLEEP_SECONDS)
    counts.to_csv(cache_path, index=False)
counts.head(16)


## 3. Audit the real source table


In [ ]:
audit = source_audit_table(counts, value_col="count", entity_col="series", time_col="month")
audit


## 4. Decompose counts with De-Time

For monthly counts, period 12 is the first cycle to test. The method is `MA_BASELINE` by default because it is built into De-Time and keeps the tutorial dependency-light.


In [ ]:
components = decompose_table(
    counts,
    entity_col="series",
    time_col="month",
    value_col="count",
    method="MA_BASELINE",
    period=12,
    trend_window=5,
    transform="log1p",
)
components.head(16)


## 5. Rank fields by trend and residual shock


In [ ]:
summary = component_summary(components, entity_col="series", time_col="month")
priority = editorial_priority(summary, entity_col="series")
priority


## 6. Top residual events

These rows are article hooks. They do not prove causality; they mark months that do not fit the smooth trend/cycle baseline.


In [ ]:
events = residual_event_table(components, entity_col="series", time_col="month", top_n=20)
events


## 7. Article-ready language


In [ ]:
guardrails = article_language_guardrails()
guardrails


In [ ]:
save_table(watchlist, "01_arxiv_category_watchlist")
save_table(audit, "01_arxiv_category_audit")
save_table(priority, "01_arxiv_category_priority")
save_table(events, "01_arxiv_category_residual_events")
save_table(guardrails, "01_arxiv_category_guardrails")
